# Sesión 05: Práctica con Python y Pandas

-----

In [1]:
import pandas as pd

In [3]:
ventas = pd.read_csv("data/ventas.csv")

### **Caso práctico:**

Se tiene una base de datos de los clientes de una empresa con su comportamiento de compra en los meses de Enero a Abril del 2020 y 2021, con la siguiente información:

- id: Número único de registro del cliente.

- ciudad: Ciudad donde se encuentra la tienda.

- edad: Edad actual.

- genero: Género (Female / Male).

- mes: Mes en el que se realizó la orden.

- ordenes: Número de órdenes generadas en el mes.

- paid: Variable que indica si las órdenes fueron pagadas o no (1 Si, 0 No)

- usd: Dólares del total de órdenes generadas.

### Ejercicio 1

- Asigna a cada uno de los IDs su generación de edad en una nueva columna en base a lo siguiente.
    - Gen Z: 18 - 25
    - Millenials: 26 - 41
    - Gen X: 42 - 57
    - Boomers: 58 - 67

- Calcula por periodo:
    - Total de órdenes mensuales
    - Promedio mensual de USD (Total de USD por mes / Total de órdenes)
    - Total de USD

- Contesta las siguientes preguntas:
    - ¿Quién es el grupo de edad con más ventas totales después del 2021?
    - ¿Cuántas órdenes hay sin pagar del mes de Febrero de 2021?


### 0 

Comienza viendo tu dataset

In [5]:
ventas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       4000 non-null   int64 
 1   ciudad   4000 non-null   object
 2   edad     4000 non-null   int64 
 3   genero   4000 non-null   object
 4   mes      4000 non-null   object
 5   ordenes  4000 non-null   int64 
 6   pago     4000 non-null   int64 
 7   usd      4000 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 250.1+ KB


### 1.1 Asignar los grupos de edad. 

**HINT**: Usar función `pd.cut()`

In [7]:
age_bins = [
    17,25,41,57,67
]
age_labels = [
    "Gen Z","Millenials","Gen X","Boomers"
]
ventas["grupo_edad"] = pd.cut(ventas["edad"], bins=age_bins, labels=age_labels, right=True)

### 1.2 Calcular para los meses de 2021: 

**HINT**: Es necesario primero convertir la columna 'mes' a datetime. 

- Total de órdenes mensuales

- Promedio mensual de USD (Total de USD por mes / Total de órdenes)

- Total de USD

In [9]:
ventas["mes_dt"] = pd.to_datetime(ventas["mes"], format='%b %y')

- Total de órdenes mensuales

In [11]:
ventas.loc[ventas["mes_dt"]>="2021-01-01"].groupby("mes_dt")["ordenes"].sum()

mes_dt
2021-01-01    1484
2021-02-01    1242
2021-03-01    1556
2021-04-01    1310
Name: ordenes, dtype: int64

- Promedio mensual de USD por orden (Suma de USD / Total de órdenes)

In [15]:
suma_por_mes = ventas.loc[ventas["mes_dt"]>="2021-01-01"].groupby("mes_dt")["usd"].sum() 
ordenes_por_mes = ventas.loc[ventas["mes_dt"]>="2021-01-01"].groupby("mes_dt")["ordenes"].sum()

usd_por_orden_por_mes = suma_por_mes / ordenes_por_mes

usd_por_orden_por_mes

mes_dt
2021-01-01    195.320081
2021-02-01    157.739130
2021-03-01    204.662596
2021-04-01    150.182443
dtype: float64

- Total de USD por mes

In [17]:
ventas.loc[ventas["mes_dt"]>="2021-01-01"].groupby("mes_dt")["usd"].sum() 

mes_dt
2021-01-01    289855
2021-02-01    195912
2021-03-01    318455
2021-04-01    196739
Name: usd, dtype: int64

### 1.3 Cálculo después de 2021:

- ¿Quién es el grupo de edad con más ventas totales después del 2021?

- ¿Cuántas órdenes hay sin pagar del mes de Febrero de 2021?

- ¿Quién es el grupo de edad con más ventas totales después del 2021?

In [29]:
ventas.head()

,id,ciudad,edad,genero,mes,ordenes,pago,usd,grupo_edad,mes_dt,ordenes_guadalajara
0,1,Guadalajara,33,Male,Jan 20,0,0,0,Millenials,2020-01-01,0.00
1,2,Guadalajara,23,Female,Jan 20,3,1,327,Gen Z,2020-01-01,379.32
2,3,Monterrey,23,Female,Jan 20,1,0,28,Gen Z,2020-01-01,28.00
3,4,Leon,24,Female,Jan 20,0,0,0,Gen Z,2020-01-01,0.00
4,5,Puebla,46,Male,Jan 20,2,1,456,Gen X,2020-01-01,456.00


In [33]:
ventas[ventas["mes_dt"]>="2021-01-01"].groupby(["grupo_edad"])["usd"].sum().sort_values(ascending=False)

/var/folders/5_/9lpts4396xlf9_gxc7xwyx140000gn/T/ipykernel_84395/4201706120.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ventas[ventas["mes_dt"]>="2021-01-01"].groupby(["grupo_edad"])["usd"].sum().sort_values(ascending=False)


grupo_edad
Millenials    536919
Gen X         296586
Gen Z         120494
Boomers        46962
Name: usd, dtype: int64

- ¿Cuántas órdenes hay sin pagar del mes de Febrero de 2021?

In [39]:
ventas[(ventas["mes_dt"]=="2021-02-01") & (ventas["pago"] == 0)]["ordenes"].sum()

92

- Agrega un impuesto del 16% a aquellas órdenes que sean compradas en Guadalajara

**HINT**: Usa  `apply` y `lambda` sobre las columnas de `usd` y `ciudad`.

Por ejemplo: 

`df[['columna_a','columna_b']].apply(lambda x: 1 if x['columna_a'] > x['columna_b'] else 0, axis = 1)`

`axis = 1` se usa cuando aplicas la función sobre más de una columna

In [23]:
ventas["ordenes_guadalajara"] = ventas[["ciudad","usd"]].apply(lambda x: x["usd"]*1.16 if x["ciudad"] == "Guadalajara" else x["usd"], axis = 1)

In [25]:
ventas.head()

,id,ciudad,edad,genero,mes,ordenes,pago,usd,grupo_edad,mes_dt,ordenes_guadalajara
0,1,Guadalajara,33,Male,Jan 20,0,0,0,Millenials,2020-01-01,0.00
1,2,Guadalajara,23,Female,Jan 20,3,1,327,Gen Z,2020-01-01,379.32
2,3,Monterrey,23,Female,Jan 20,1,0,28,Gen Z,2020-01-01,28.00
3,4,Leon,24,Female,Jan 20,0,0,0,Gen Z,2020-01-01,0.00
4,5,Puebla,46,Male,Jan 20,2,1,456,Gen X,2020-01-01,456.00
